# 02. Três métodos de seleção e votação das features

Os três métodos rodam aqui, no mesmo notebook, e todos enxergam só o treino. O pré-processamento (imputação e padronização) também é ajustado só no treino. Como não usamos validação cruzada, não existe seleção dentro de fold: a validação e o teste simplesmente não entram nesta etapa.

- **SelectKBest com `f_regression`:** teste F de cada atributo contra o alvo, um de cada vez. Mede relação linear e é a estatística adequada para alvo contínuo.
- **Importância do Random Forest:** quanto cada atributo reduz o erro nas divisões das árvores. Pega relações não lineares e interações.
- **RFE com Ridge:** tira, um por vez, o atributo de menor coeficiente até sobrarem k. Olha os atributos em conjunto, então descarta os redundantes.

**Regra de votação, fixada no `config.yaml` antes de rodar:** cada método vota nas suas k = 10 melhores. É eleita a feature com pelo menos 2 dos 3 votos. Se a maioria eleger menos de 6, o conjunto é completado pelo menor ranking médio dos três métodos, que também desempata a ordem. Esse conjunto é o único usado pelos dois modelos do ciclo.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src import ciclo, features, preprocessamento

cfg = ciclo.carregar_config()
estado = ciclo.atual()
mlflow = ciclo.configurar_mlflow(cfg)
dataset, janelas = preprocessamento.carregar(cfg)
pd.set_option("display.precision", 3)
print(f"ciclo {estado['cycle_id']} | {estado['data_version']}")

In [ ]:
treino = janelas["treino"]
candidatas = [c for c in dataset.columns if c != "alvo"]
print(f"{len(candidatas)} candidatas, treino de {treino.index.min():%d/%m/%Y} a {treino.index.max():%d/%m/%Y} ({len(treino)} dias)")

ranking = features.tres_metodos(treino, candidatas, cfg, cfg["projeto"]["semente"])
ranking[["f_score", "rank_kbest", "rf_importancia", "rank_rf", "rfe_ranking", "rank_rfe"]].sort_values("rank_rf")

## Tabela de votos

In [ ]:
votos, eleitas = features.votar(ranking, cfg)
print(f"eleitas ({len(eleitas)}):", ", ".join(eleitas))
votos[["rank_kbest", "rank_rf", "rank_rfe", "voto_kbest", "voto_rf", "voto_rfe", "votos", "ranking_medio", "eleita"]]

## Registro no MLflow

Um run de votação com os parâmetros da regra, as features eleitas, a tabela de votos completa (com rankings e importâncias) e as tags `cycle_id` e `data_version` do ciclo.

In [ ]:
v = cfg["votacao"]
arquivo_votos = ciclo.pasta(estado) / "votos.csv"
votos.to_csv(arquivo_votos, index_label="feature")
with mlflow.start_run(run_name=f"{estado['cycle_id']}-votacao", tags=ciclo.tags(estado, "votacao")) as run:
    mlflow.log_params({"metodos": "selectkbest_f_regression,random_forest,rfe_ridge", "metodo_votacao": v["metodo"],
                       "k": v["k"], "votos_minimos": v["votos_minimos"], "min_features": v["min_features"],
                       "desempate": v["desempate"], "rf_arvores": v["rf_arvores"], "semente": cfg["projeto"]["semente"],
                       "features_eleitas": ",".join(eleitas)})
    mlflow.log_metric("n_eleitas", len(eleitas))
    mlflow.log_artifact(arquivo_votos)

estado["votacao"] = {"run_id": run.info.run_id, "features": eleitas, "n": len(eleitas)}
ciclo.salvar(estado)
print("run de votação:", run.info.run_id)